# EoMT LoRA Fine-Tuning (10 epochs) + OOD Evaluation

This notebook fine-tunes EoMT with LoRA for 10 epochs and then runs MSP, MaxLogit, and MaxEntropy on anomaly datasets.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Set your paths. `DATA_DIR` must contain the Cityscapes zip files (`leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip`).

In [ ]:
import os
from pathlib import Path

CODE_DIR = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject"  # TODO: change
DATA_DIR = "/content/drive/MyDrive/datasets"  # TODO: change
OOD_INPUT_GLOB = "/content/drive/MyDrive/anomaly/RoadObsticle21/images/*.webp"  # TODO: change

CKPT_PATH = os.path.join(CODE_DIR, "eomt/checkpoints/eomt_cityscapes.bin")

In [ ]:
os.chdir(CODE_DIR)
print("Working dir:", os.getcwd())

Install dependencies. This keeps the Colab-provided torch/torchvision versions.

In [ ]:
import pathlib

reqs = pathlib.Path("eomt/requirements.txt").read_text().splitlines()
filtered = [r for r in reqs if not r.startswith("torch==") and not r.startswith("torchvision==")]
pathlib.Path("/tmp/eomt_requirements.txt").write_text("\n".join(filtered))

!pip install -r /tmp/eomt_requirements.txt

Create a LoRA fine-tuning config (10 epochs). Adjust batch size if you hit OOM.

In [ ]:
import textwrap

lora_config = f'''
trainer:
  max_epochs: 10
  default_root_dir: "outputs"
  logger: false
  log_every_n_steps: 1
  enable_progress_bar: true
  enable_checkpointing: true
  callbacks:
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        dirpath: "outputs/checkpoints"
        save_last: true
        save_top_k: 0
model:
  init_args:
    ckpt_path: "{CKPT_PATH}"
    lora_enabled: true
    lora_r: 8
    lora_alpha: 16.0
    lora_dropout: 0.05
    lora_target_modules: ["qkv", "proj"]
    lora_train_bias: "none"
    lora_trainable_modules: ["class_head", "mask_head", "q"]
data:
  init_args:
    path: "{DATA_DIR}"
    batch_size: 1
    num_workers: 2
'''

Path("lora_finetune.yaml").write_text(textwrap.dedent(lora_config))
print(Path("lora_finetune.yaml").read_text())

Run LoRA fine-tuning for 10 epochs.

In [ ]:
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

!python eomt/main.py fit \
  -c eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  -c lora_finetune.yaml \
  --compile_disabled

Find the latest checkpoint produced by the run.

In [ ]:
import glob

ckpt_candidates = sorted(
    glob.glob("outputs/checkpoints/*.ckpt"),
    key=os.path.getmtime,
)
if not ckpt_candidates:
    ckpt_candidates = sorted(
        glob.glob("outputs/**/checkpoints/*.ckpt", recursive=True),
        key=os.path.getmtime,
    )
if not ckpt_candidates:
    raise FileNotFoundError("No checkpoints found under outputs/**/checkpoints")
FINETUNED_CKPT = ckpt_candidates[-1]
print("Using checkpoint:", FINETUNED_CKPT)

Evaluate MSP, MaxLogit, and MaxEntropy. Update `OOD_INPUT_GLOB` to point at each dataset split you want to test.

In [ ]:
import subprocess

methods = ["msp", "maxlogit", "maxentropy"]
for method in methods:
    cmd = [
        "python",
        "eomt/evalAnomaly.py",
        "--input",
        OOD_INPUT_GLOB,
        "--ckpt",
        FINETUNED_CKPT,
        "--method",
        method,
        "--lora",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()